# Module 2.2: Clamshell Lid-Close Operation, Thermal Capping, SSH Hardening & Raspberry Pi NVMe Cleanup

## Homelab Expansion (Chapter 2) - Step 2: Operating System & Headless Laptop Tuning

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Configure `systemd-logind` so closing the laptop lid (`HandleLidSwitch=ignore`) keeps `homelab-gpu` running 24/7 instead of suspending to RAM.
2. Enable Linux kernel `consoleblank=60` to power down the laptop's LCD backlight after 60 seconds and respect the **ASUS ROG Zephyrus ErgoLift keyboard intake airflow**.
3. Install `lm-sensors` and `nvme-cli` to verify post-repaste CPU/GPU temperatures and native PCIe SMART health on both internal NVMe drives.
4. Harden `sshd_config` on `homelab-gpu` using your existing `id_ed25519_homelab` SSH key from Chapter 1.
5. **Decommission the migrated NVMe SSD and old Immich stack from `homelab-pi`**—stripping stale `/etc/fstab` mount and swapfile entries and proving the Raspberry Pi reboots cleanly with zero external drives attached before Step 3 joins both nodes onto the enterprise switch.

## 2.2.1 Conceptual Background: Turning a Gaming Laptop Into a Headless Appliance & Cleaning Up `homelab-pi`

Unlike a Raspberry Pi or rackmount server, a laptop has three built-in hardware behaviors that work *against* 24/7 server operation unless explicitly overridden in Linux:

### 1. The Magnetic Hall-Effect Lid Switch (`systemd-logind`)
Laptops contain a tiny magnet in the top display bezel and a Hall-effect sensor in the palmrest. By default, when you close the lid on Ubuntu Server, `systemd-logind` intercepts the ACPI lid event and triggers `suspend` (putting the CPU to sleep and dropping all SSH/Docker connections). Overriding `HandleLidSwitch=ignore` in `/etc/systemd/logind.conf` disables sleep completely.

### 2. The LCD Backlight & Console Blanking (`consoleblank=60`)
When `HandleLidSwitch=ignore` is active (or when the lid is left slightly ajar for airflow), the Linux TTY text console stays illuminated 24/7 unless instructed to blank via VESA DPMS. Passing `consoleblank=60` to the GRUB kernel command line powers down the display panel and backlight 60 seconds after the last physical keypress, dropping idle wall power draw by **4W to 7W**.

### 3. Crucial ROG Zephyrus Airflow Nuance: Keyboard Deck Intake & ErgoLift
The 2019 ASUS ROG Zephyrus (`GA502` / `GU502` / `GX502`) was engineered with two unique thermal features:
1. **Top-Deck Keyboard Air Intake:** The blower fans draw cool air in through both the bottom vents *and* down through the keyboard deck / perforated grill above the function keys.
2. **ErgoLift Hinge:** Opening the lid angles the rear of the laptop up by several millimeters to increase bottom intake clearance.
> 💡 **Best Practice Physical Placement:** Even after setting `HandleLidSwitch=ignore`, either leave the laptop lid propped open at a **15°–30° angle** (putting a small rubber bumper under the rear feet) or place the laptop in a vertical/elevated cooling stand so cool air flows freely over the keyboard deck while the screen remains completely powered off via `consoleblank=60`!

### 4. Why We Clean Up `homelab-pi` Right Now (Before Step 3 Networking)
Now that `homelab-gpu` is powered on, hardened, and safely holding your WD Blue SN5000 + `immich_migration_dump.sql.gz` internally, we should immediately clean up the Raspberry Pi's leftover Chapter 1 storage references before wiring both nodes onto the `USW-Enterprise-8-PoE` switch:
- **Prevent Stale `/etc/fstab` Warnings & Boot Delays:** In Chapter 1, we added two entries to `/etc/fstab` on `homelab-pi`: the `UUID=... /mnt/homelab_storage` mount and `/mnt/homelab_storage/swapfile`. Even with `nofail`, leaving missing device entries causes `systemd` boot delays and failed mount unit warnings.
- **Prevent Silent MicroSD Corruption:** If the empty `/mnt/homelab_storage` directory is left behind on the Pi, any misconfigured container or script writing to it would silently fill the MicroSD root partition (`/`).
- **Reclaim Gigabytes of MicroSD Space:** Pruning the stopped Chapter 1 Immich ARM64 container images frees ~2–3 GB on the Pi's MicroSD card, leaving `homelab-pi` ultra-lean for its dedicated **Tier-0 Network & Monitoring** role (Pi-hole v6, Tailscale, Uptime Kuma) where total RAM usage sits under ~450 MB out of 4 GB (no disk swapfile required!).

## 2.2.2 Step-by-Step Laboratory Guide

### Task 2.2.1: Disable Lid-Close Suspend in `systemd-logind` (`homelab-gpu`)
Connect to `homelab-gpu` via SSH and configure `systemd-logind` to ignore lid-close events under all power states:
```bash
sudo tee -a /etc/systemd/logind.conf > /dev/null << 'EOT'
# Homelab 24/7 Server Mode: Never suspend when laptop lid is closed
HandleLidSwitch=ignore
HandleLidSwitchExternalPower=ignore
HandleLidSwitchDocked=ignore
LidSwitchIgnoreInhibited=no
EOT

# Apply immediately without rebooting
sudo systemctl restart systemd-logind
```

### Task 2.2.2: Enable 60-Second LCD Backlight Off (`consoleblank=60`)
Configure GRUB so the laptop screen turns completely off after 60 seconds of keyboard inactivity:
```bash
# 1. Add consoleblank=60 to GRUB_CMDLINE_LINUX_DEFAULT
sudo sed -i 's/GRUB_CMDLINE_LINUX_DEFAULT="/GRUB_CMDLINE_LINUX_DEFAULT="consoleblank=60 /' /etc/default/grub

# 2. Regenerate the GRUB boot configuration
sudo update-grub

# 3. Also trigger immediate 60-second blanking on the current active TTY1 console
sudo sh -c 'TERM=linux setterm --blank 1 --powerdown 1 </dev/tty1 >/dev/tty1' 2>/dev/null || true
```
Verify that the kernel parameter is queued in `/boot/grub/grub.cfg`:
```bash
grep "consoleblank=60" /boot/grub/grub.cfg
```

### Task 2.2.3: Install Hardware Sensors & Verify Dual-NVMe SMART Health (`homelab-gpu`)
Install `lm-sensors` and `nvme-cli` to monitor both post-repaste CPU/GPU temperatures and native PCIe NVMe health on your stock OS drive and migrated WD Blue SN5000:
```bash
# 1. Update apt cache, upgrade packages, and install thermal/NVMe telemetry tools
sudo apt update && sudo apt full-upgrade -y
sudo apt install -y lm-sensors linux-tools-common linux-tools-generic nvme-cli btop

# 2. Auto-detect motherboard/CPU/NVMe thermal sensors
sudo sensors-detect --auto

# 3. Inspect live CPU, dual-NVMe SSD, and ASUS motherboard temperatures
sensors
sudo nvme list
```
Verify that idle CPU and NVMe temperatures sit comfortably around **35°C – 48°C** thanks to your fresh thermal paste and `K5 PRO` putty from Module 1.2!

### 🖼️ Task 2.2.3 Verification: Thermal Sensor, Dual-NVMe & Lid-Close Check

**Verification Objective:** Capture a screenshot of your SSH session on `homelab-gpu` showing `sensors`, `sudo nvme list` (both internal NVMe drives visible), and `tail -n 5 /etc/systemd/logind.conf`.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

### Task 2.2.4: Copy Your Homelab Ed25519 SSH Key & Harden `sshd` (`homelab-gpu`)
In Chapter 1, you generated `~/.ssh/id_ed25519_homelab` on your workstation. Authorize that exact same cryptographic key on `homelab-gpu` and disable password-based SSH logins:

1. **From your Workstation Terminal** (macOS / Linux / Windows PowerShell), copy your public key to `homelab-gpu`:
   - **macOS / Linux / Git Bash:**
     ```bash
     ssh-copy-id -i ~/.ssh/id_ed25519_homelab.pub pi@homelab-gpu.local
     ```
   - **Windows PowerShell:**
     ```powershell
     type $env:USERPROFILE\.ssh\id_ed25519_homelab.pub | ssh pi@homelab-gpu.local "mkdir -p ~/.ssh && chmod 700 ~/.ssh && cat >> ~/.ssh/authorized_keys && chmod 600 ~/.ssh/authorized_keys"
     ```
2. **Test Key-Based Login from your Workstation:**
   ```bash
   ssh -i ~/.ssh/id_ed25519_homelab pi@homelab-gpu.local
   ```
3. **Once logged into `homelab-gpu` with your key**, lock down SSH via a drop-in configuration file:
   ```bash
   sudo tee /etc/ssh/sshd_config.d/99-homelab-hardening.conf > /dev/null << 'EOT'
   PasswordAuthentication no
   KbdInteractiveAuthentication no
   PermitRootLogin no
   PubkeyAuthentication yes
   EOT

   sudo sshd -t && sudo systemctl reload ssh
   ```

### Task 2.2.5: Clean Up `homelab-pi` & Verify Clean Boot Without the NVMe SSD
Now that `homelab-gpu` is powered on, hardened, and safely holding `/mnt/homelab_storage` internally—and **before** we wire `homelab-gpu` into the homelab switch and Tailnet in Step 3—let's cleanly strip the old Immich stack and external NVMe references off **`homelab-pi`** so the Raspberry Pi boots cleanly forever without any USB drive attached!

1. **SSH into `homelab-pi`** (`ssh homelab-pi`) and remove the retired Chapter 1 Immich stack and cached ARM64 images:
   ```bash
   # 1. Confirm Immich containers are stopped and remove the old Pi Immich stack directory
   if [ -d ~/homelab/stacks/immich ]; then
       cd ~/homelab/stacks/immich && docker compose down --remove-orphans
       cd ~ && rm -rf ~/homelab/stacks/immich
   fi

   # 2. Prune unused Immich/Postgres/ML ARM64 container images to reclaim ~2-3 GB on the MicroSD card
   docker image prune -a -f
   ```
2. **Back up `/etc/fstab` and remove BOTH `/mnt/homelab_storage` lines** (the Chapter 1 Module 3.2 `UUID` mount AND the Chapter 1 Module 5.1 `/mnt/homelab_storage/swapfile` line):
   ```bash
   # 3. Save a timestamped backup before editing
   sudo cp /etc/fstab /etc/fstab.pre-nvme-removal.$(date +%F)

   # 4. Remove any lines referencing /mnt/homelab_storage (mount + swapfile)
   sudo sed -i '\|/mnt/homelab_storage|d' /etc/fstab

   # 5. Inspect /etc/fstab to confirm ONLY the MicroSD boot/root partitions remain!
   cat /etc/fstab
   ```
3. **Remove the stale empty mount point, reload `systemd`, and reboot `homelab-pi` without the NVMe plugged in:**
   ```bash
   # 6. Confirm nothing is mounted there and remove the stale directory so nothing writes to SD accidentally
   mountpoint /mnt/homelab_storage || sudo rmdir /mnt/homelab_storage 2>/dev/null || true
   sudo systemctl daemon-reload

   # 7. Reboot homelab-pi to prove clean standalone boot without the NVMe SSD!
   sudo reboot
   ```
4. Wait ~30 seconds, SSH back into `homelab-pi` (`ssh homelab-pi`), and run the **standalone boot health check**:
   ```bash
   # Expect 0 failed systemd units, clean df -h, and ~3.6 GB+ free RAM!
   systemctl --failed
   df -h /
   free -h
   docker ps -a
   ```
   `homelab-pi` is now a spotless, ultra-lean **5W Tier-0 network & monitoring platform** ready to join the `USW-Enterprise-8-PoE` cluster!

### 🖼️ Task 2.2.5 Verification: Hardened `homelab-gpu` & Clean Standalone Reboot of `homelab-pi`

**Verification Objective:** Capture a screenshot showing either (a) passwordless SSH + `btop` on `homelab-gpu`, and/or (b) `homelab-pi` cleanly rebooted without the NVMe (`systemctl --failed` showing `0 loaded units listed`, clean `cat /etc/fstab`, and `free -h`).

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] `HandleLidSwitch=ignore` active in `/etc/systemd/logind.conf` and `consoleblank=60` configured in GRUB on `homelab-gpu`.
- [ ] `lm-sensors` and `nvme-cli` installed on `homelab-gpu`, confirming healthy idle temps and both internal NVMe drives.
- [ ] Workstation `id_ed25519_homelab` SSH key authorized on `homelab-gpu` and password SSH authentication disabled.
- [ ] Old Chapter 1 Immich stack, unused ARM64 images, and **both** `/mnt/homelab_storage` + `swapfile` `/etc/fstab` entries cleanly removed from `homelab-pi`.
- [ ] `homelab-pi` rebooted **without** the NVMe SSD attached and verified clean (`0 failed units` in `systemctl --failed`).
- [ ] Verification screenshots captured and attached above.

Proceed to [Module 3.1: UniFi Switch Enterprise 8 PoE Topology, Static IP Plan & Unified Mesh](../03_network_expansion_and_cluster_topology/03_1_switch_topology_and_subnet_design.ipynb).